# Task 2: LoRA Fine-Tuning of Llama 3.2 1B for Named Entity Recognition

## Internship Experiment 2

This notebook performs parameter-efficient fine-tuning (LoRA) on Llama 3.2 1B for a token classification (NER) task.

### Objectives

- Load and validate the dataset
- Perform tokenization and label alignment
- Configure LoRA adapters
- Fine-tune the model
- Evaluate model performance
- Compare with baseline results

In [1]:
!pip install -q transformers datasets evaluate seqeval peft bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00


# 1. Environment Setup

In [2]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
from transformers import set_seed

set_seed(42)

print("Random seed set to 42")

Random seed set to 42


In [4]:
import os
import json
import numpy as np
import torch
import evaluate

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)